# 01 · API Exploration: Jolpica F1 API

**Blueprint step:** Step 2 (API exploration) · **Checkpoint:** part of Checkpoint 1

**Purpose:** check the real behaviour of the Jolpica F1 API **by experiment** before building the reusable cached client (Step 3). We look at real responses and write down what we find. Nothing here is guessed.

**This notebook does NOT:** collect 1950–2025 data, build `src/api_client.py`, fetch weather, engineer features or train models.

| Part | What we check |
|---|---|
| 0 | Setup: load `config.yaml`, a tiny request helper |
| 1 | Connectivity and response structure (2024 Round 1 results) |
| 2 | Pagination (`limit`, `offset`, `MRData.total`) using the 2024 season only |
| 3 | 2024 Round 1 as a tidy DataFrame |
| 4 | Qualifying endpoint |
| 5 | Sprint endpoint |
| 6 | Edge cases and data behaviour (DNF, grid = 0, ID stability) |
| 7 | Response headers and rate limits |
| 8 | Raw responses saved to disk |
| 9 | Findings summary |

**How to run:** select the project's virtual environment as the kernel and use *Run All*. About 35 API requests are made, spaced 1 s apart (≈ 1 minute). Responses are fetched live each run; caching comes in Step 3.

## 0. Setup

We find the project root by walking up from the current folder until we see `config.yaml`. This way the notebook works from any folder and **no absolute Windows paths** are used. All API settings (base URL, timeout, spacing between requests, User-Agent) come from `config.yaml`.

In [1]:
import json
import time
from pathlib import Path

import pandas as pd
import requests
import yaml

# Locate the project root (the folder that contains config.yaml)
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "config.yaml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("config.yaml not found - open this notebook inside the project")
    PROJECT_ROOT = PROJECT_ROOT.parent

with open(PROJECT_ROOT / "config.yaml", encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)

BASE_URL = CONFIG["api"]["jolpica"]["base_url"]
TIMEOUT = CONFIG["api"]["timeout_seconds"]
WAIT_SECONDS = CONFIG["api"]["jolpica"]["min_seconds_between_requests"]
HEADERS = {"User-Agent": CONFIG["api"]["user_agent"]}

# Small exploratory responses are kept apart from the Step 4 collection folder
EXPLORATION_DIR = PROJECT_ROOT / CONFIG["paths"]["data_raw"] / "api_exploration"
EXPLORATION_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# Make sure we run in the project's virtual environment (versions pinned in requirements.txt)
import sys
from importlib.metadata import version
pins = dict(line.split("==") for line in (PROJECT_ROOT / "requirements.txt").read_text().splitlines()
            if "==" in line and not line.startswith("#"))
for package in ["pandas", "requests", "PyYAML"]:
    assert version(package) == pins[package], (
        f"{package} {version(package)} != pinned {pins[package]}: "
        "select the 'Python 3.11 (f1-race-prediction)' kernel")

print("Python       :", sys.version.split()[0], "|", Path(sys.executable).parent.parent.name)
print("pandas", pd.__version__, "| requests", requests.__version__, "(match requirements.txt)")
print("Project root :", PROJECT_ROOT.name)
print("Base URL     :", BASE_URL)
print("Wait between requests:", WAIT_SECONDS, "s | timeout:", TIMEOUT, "s")
print("Raw exploration folder:", EXPLORATION_DIR.relative_to(PROJECT_ROOT))

Python       : 3.11.9 | f1-race-prediction
pandas 2.3.1 | requests 2.32.4 (match requirements.txt)
Project root : F1-Race-Prediction
Base URL     : https://api.jolpi.ca/ergast/f1
Wait between requests: 1.0 s | timeout: 30 s
Raw exploration folder: data\raw\api_exploration


**Request helper.** This is deliberately tiny and is **not** the Step 3 client: no caching and no retries. It only:
1. waits `WAIT_SECONDS` before each call (to stay polite to the API),
2. sends a GET request with our User-Agent,
3. optionally saves the **exact raw bytes** of a successful response,
4. logs every request so we can count them.

In [2]:
REQUEST_LOG = []

def get(path, params=None, save_as=None):
    # Send one GET request to the Jolpica API and return the requests.Response.
    time.sleep(WAIT_SECONDS)
    url = BASE_URL + path
    response = requests.get(url, params=params, headers=HEADERS, timeout=TIMEOUT)
    REQUEST_LOG.append({
        "url": response.url.replace(BASE_URL, ""),
        "status": response.status_code,
        "seconds": round(response.elapsed.total_seconds(), 2),
        "saved_as": save_as,
    })
    if save_as and response.status_code == 200:
        (EXPLORATION_DIR / save_as).write_bytes(response.content)  # exact bytes, unmodified
    return response


def mrdata_summary(js):
    # Return the scalar (non-nested) fields of MRData, e.g. limit, offset, total.
    return {k: v for k, v in js["MRData"].items() if not isinstance(v, dict)}

## 1. API connectivity: 2024 Round 1 results

Endpoint: `/2024/1/results.json`. We request `limit=100` so the whole race arrives in one page.

In [3]:
r = get("/2024/1/results.json", params={"limit": 100}, save_as="2024_r01_results.json")

print("1. HTTP status code :", r.status_code)
print("2. Content-Type     :", r.headers.get("Content-Type"))
results_r1 = r.json()  # raises an error if the body is not valid JSON
print("3. JSON parsed      : OK (type:", type(results_r1).__name__ + ")")
print("4. Top-level keys   :", list(results_r1.keys()))
print("5. MRData keys      :", list(results_r1["MRData"].keys()))
print("   MRData scalars   :", mrdata_summary(results_r1))
print("6. MRData.total     :", results_r1["MRData"]["total"], "(a string, not a number)")

1. HTTP status code : 200
2. Content-Type     : application/json
3. JSON parsed      : OK (type: dict)
4. Top-level keys   : ['MRData']
5. MRData keys      : ['xmlns', 'series', 'url', 'limit', 'offset', 'total', 'RaceTable']
   MRData scalars   : {'xmlns': '', 'series': 'f1', 'url': 'https://api.jolpi.ca/ergast/f1/2024/1/results.json', 'limit': '100', 'offset': '0', 'total': '20'}
6. MRData.total     : 20 (a string, not a number)


In [4]:
race_table = results_r1["MRData"]["RaceTable"]
race = race_table["Races"][0]
first = race["Results"][0]

print("RaceTable keys :", list(race_table.keys()))
print("Races returned :", len(race_table["Races"]))
print()
print("7. RACE keys   :", list(race.keys()))
print("   season/round:", race["season"], "/", race["round"], "|", race["raceName"], "|", race["date"], race.get("time"))
print("   Circuit     :", {k: v for k, v in race["Circuit"].items() if k != "url"})
print()
print("10. RESULT keys (first record):", list(first.keys()))
print("8.  DRIVER keys     :", list(first["Driver"].keys()))
print("9.  CONSTRUCTOR keys:", list(first["Constructor"].keys()))
print()
print("Example result (winner):")
print("  driver     :", first["Driver"]["driverId"], first["Driver"].get("code"), first["Driver"]["givenName"], first["Driver"]["familyName"])
print("  constructor:", first["Constructor"]["constructorId"], "/", first["Constructor"]["name"])
print("  grid:", first["grid"], "| position:", first["position"], "| positionText:", first["positionText"],
      "| points:", first["points"], "| laps:", first["laps"], "| status:", first["status"])
print("  Time       :", first.get("Time"))
print("  FastestLap :", first.get("FastestLap"))
print()
print("Number of result records:", len(race["Results"]))

RaceTable keys : ['season', 'round', 'Races']
Races returned : 1

7. RACE keys   : ['season', 'round', 'url', 'raceName', 'Circuit', 'date', 'time', 'Results']
   season/round: 2024 / 1 | Bahrain Grand Prix | 2024-03-02 15:00:00Z
   Circuit     : {'circuitId': 'bahrain', 'circuitName': 'Bahrain International Circuit', 'Location': {'lat': '26.0325', 'long': '50.5106', 'locality': 'Sakhir', 'country': 'Bahrain'}}

10. RESULT keys (first record): ['number', 'position', 'positionText', 'points', 'Driver', 'Constructor', 'grid', 'laps', 'status', 'Time', 'FastestLap']
8.  DRIVER keys     : ['driverId', 'permanentNumber', 'code', 'url', 'givenName', 'familyName', 'dateOfBirth', 'nationality']
9.  CONSTRUCTOR keys: ['constructorId', 'url', 'name', 'nationality']

Example result (winner):
  driver     : max_verstappen VER Max Verstappen
  constructor: red_bull / Red Bull
  grid: 1 | position: 1 | positionText: 1 | points: 26 | laps: 57 | status: Finished
  Time       : {'millis': '5504742', 't

**Observation:** every value, including numbers such as `grid`, `position`, `points` and `total`, arrives as a **string**. Type conversion will be needed during cleaning (Step 8).

## 2. Pagination

We use the **2024 season results** (`/2024/results.json`), which has a few hundred rows. That is enough to test paging without touching historical data.

A small helper turns a response into a list of `(round, driverId)` keys, so we can compare pages exactly.

In [5]:
def record_keys(js, record_key="Results"):
    # List of (round, driverId) for every record in the response, in order.
    keys = []
    for rc in js["MRData"]["RaceTable"]["Races"]:
        for rec in rc.get(record_key, []):
            keys.append((int(rc["round"]), rec["Driver"]["driverId"]))
    return keys

pagination_rows = []
def page_check(label, params):
    resp = get("/2024/results.json", params=params)
    js = resp.json()
    keys = record_keys(js)
    info = mrdata_summary(js)
    pagination_rows.append({"test": label, "requested": str(params), "MRData.limit": info["limit"],
                            "MRData.offset": info["offset"], "MRData.total": info["total"],
                            "records returned": len(keys)})
    return keys, js

keys_default, _ = page_check("default (no limit)", None)
keys_10_a, _ = page_check("limit=10, offset=0", {"limit": 10, "offset": 0})
keys_10_b, _ = page_check("limit=10, offset=10", {"limit": 10, "offset": 10})
keys_20, _ = page_check("limit=20, offset=0", {"limit": 20, "offset": 0})
keys_100, _ = page_check("limit=100, offset=0", {"limit": 100, "offset": 0})
keys_1000, _ = page_check("limit=1000 (above max)", {"limit": 1000})

pd.DataFrame(pagination_rows)

,test,requested,MRData.limit,MRData.offset,MRData.total,records returned
0,default (no limit),None,30,0,479,30
1,"limit=10, offset=0","{'limit': 10, 'offset': 0}",10,0,479,10
2,"limit=10, offset=10","{'limit': 10, 'offset': 10}",10,10,479,10
3,"limit=20, offset=0","{'limit': 20, 'offset': 0}",20,0,479,20
4,"limit=100, offset=0","{'limit': 100, 'offset': 0}",100,0,479,100
5,limit=1000 (above max),{'limit': 1000},100,0,479,100


In [6]:
print("limit respected (10 → 10, 20 → 20, 100 → 100):",
      len(keys_10_a) == 10 and len(keys_20) == 20 and len(keys_100) == 100)
print("Default limit when none given:", len(keys_default), "records")
print("limit=1000 is capped at:", len(keys_1000), "records")
print("offset changes the records (page 1 vs page 2 overlap):", len(set(keys_10_a) & set(keys_10_b)), "records in common")
print("Two pages of 10 == one page of 20 (same order):", keys_10_a + keys_10_b == keys_20)
print("Pages are contiguous with the 100-page:", keys_20 == keys_100[:20])
totals = {row["MRData.total"] for row in pagination_rows}
print("MRData.total identical in every request:", totals)

limit respected (10 → 10, 20 → 20, 100 → 100): True
Default limit when none given: 30 records
limit=1000 is capped at: 100 records
offset changes the records (page 1 vs page 2 overlap): 0 records in common
Two pages of 10 == one page of 20 (same order): True
Pages are contiguous with the 100-page: True
MRData.total identical in every request: {'479'}


**Full walk through the 2024 season** with `limit=100`, increasing `offset` by 100 until `offset >= total`. We check that:
- the number of collected records equals `MRData.total`,
- there are no duplicate `(round, driverId)` pairs,
- whether a single race is **split across two pages**, which matters for Step 4.

Each page is saved (small, and reused for the edge-case analysis in Part 6).

In [7]:
season_pages = []
offset, total = 0, None
while total is None or offset < total:
    resp = get("/2024/results.json", params={"limit": 100, "offset": offset},
               save_as=f"2024_season_results_offset_{offset:04d}.json")
    js = resp.json()
    total = int(js["MRData"]["total"])
    season_pages.append(js)
    offset += 100

all_keys = [k for js in season_pages for k in record_keys(js)]
print("Pages fetched           :", len(season_pages))
print("MRData.total            :", total)
print("Records collected       :", len(all_keys))
print("Collected == total      :", len(all_keys) == total)
print("Duplicate (round, driver):", len(all_keys) - len(set(all_keys)))

# Which rounds appear on more than one page?
rounds_per_page = [sorted({k[0] for k in record_keys(js)}) for js in season_pages]
for i, rounds in enumerate(rounds_per_page):
    print(f"  page {i} (offset {i*100:>3}): rounds {rounds[0]}–{rounds[-1]}")
split_rounds = sorted({r for i in range(len(rounds_per_page) - 1)
                       for r in set(rounds_per_page[i]) & set(rounds_per_page[i + 1])})
print("Rounds split across two pages:", split_rounds)

Pages fetched           : 5
MRData.total            : 479
Records collected       : 479
Collected == total      : True
Duplicate (round, driver): 0
  page 0 (offset   0): rounds 1–6
  page 1 (offset 100): rounds 6–11
  page 2 (offset 200): rounds 11–16
  page 3 (offset 300): rounds 16–21
  page 4 (offset 400): rounds 21–24
Rounds split across two pages: [6, 11, 16, 21]


## 3. One complete race as a DataFrame (2024 Round 1)

We flatten the nested JSON into one row per driver. Numbers are converted from text to numeric types so the table is readable. **This is not feature engineering**: it only reshapes what the API returned.

Two different "numbers" exist: `number` on the result is the **car number used in that race**, and `permanentNumber` on the driver is the **driver's permanent number**. We keep both.

In [8]:
def flatten_results(js, record_key="Results"):
    # One row per driver per race. Works for race results and sprint results.
    rows = []
    for rc in js["MRData"]["RaceTable"]["Races"]:
        for rec in rc.get(record_key, []):
            drv, con = rec["Driver"], rec["Constructor"]
            fl = rec.get("FastestLap", {})
            rows.append({
                "season": rc["season"], "round": rc["round"], "race_name": rc["raceName"],
                "race_date": rc["date"], "circuit_id": rc["Circuit"]["circuitId"],
                "driver_id": drv["driverId"], "driver_code": drv.get("code"),
                "driver_number": drv.get("permanentNumber"), "car_number": rec.get("number"),
                "driver_first_name": drv["givenName"], "driver_last_name": drv["familyName"],
                "constructor_id": con["constructorId"], "constructor_name": con["name"],
                "grid": rec.get("grid"), "position": rec.get("position"),
                "position_text": rec.get("positionText"), "points": rec.get("points"),
                "status": rec.get("status"), "laps": rec.get("laps"),
                "race_time_millis": rec.get("Time", {}).get("millis"),
                "fastest_lap_rank": fl.get("rank"), "fastest_lap_lap": fl.get("lap"),
                "fastest_lap_time": fl.get("Time", {}).get("time"),
            })
    df = pd.DataFrame(rows)
    for col in ["season", "round", "grid", "position", "points", "laps", "race_time_millis",
                "fastest_lap_rank", "fastest_lap_lap"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

df_r1 = flatten_results(results_r1)
df_r1

,season,round,race_name,race_date,circuit_id,driver_id,driver_code,driver_number,car_number,driver_first_name,driver_last_name,constructor_id,constructor_name,grid,position,position_text,points,status,laps,race_time_millis,fastest_lap_rank,fastest_lap_lap,fastest_lap_time
0,2024,1,Bahrain Grand Prix,2024-03-02,bahrain,max_verstappen,VER,3,1,Max,Verstappen,red_bull,Red Bull,1,1,1,26,Finished,57,5504742,1,39,1:32.608
1,2024,1,Bahrain Grand Prix,2024-03-02,bahrain,perez,PER,11,11,Sergio,Pérez,red_bull,Red Bull,5,2,2,18,Finished,57,5527199,4,40,1:34.364
2,2024,1,Bahrain Grand Prix,2024-03-02,bahrain,sainz,SAI,55,55,Carlos,Sainz,ferrari,Ferrari,4,3,3,15,Finished,57,5529852,6,44,1:34.507
3,2024,1,Bahrain Grand Prix,2024-03-02,bahrain,leclerc,LEC,16,16,Charles,Leclerc,ferrari,Ferrari,2,4,4,12,Finished,57,5544411,2,36,1:34.090
4,2024,1,Bahrain Grand Prix,2024-03-02,bahrain,russell,RUS,63,63,George,Russell,mercedes,Mercedes,3,5,5,10,Finished,57,5551530,12,40,1:35.065
5,2024,1,Bahrain Grand Prix,2024-03-02,bahrain,norris,NOR,1,4,Lando,Norris,mclaren,McLaren,7,6,6,8,Finished,57,5553200,5,35,1:34.476
6,2024,1,Bahrain Grand Prix,2024-03-02,bahrain,hamilton,HAM,44,44,Lewis,Hamilton,mercedes,Mercedes,9,7,7,6,Finished,57,5555066,7,39,1:34.722
7,2024,1,Bahrain Grand Prix,2024-03-02,bahrain,piastri,PIA,81,81,Oscar,Piastri,mclaren,McLaren,8,8,8,4,Finished,57,5560824,11,39,1:34.983
8,2024,1,Bahrain Grand Prix,2024-03-02,bahrain,alonso,ALO,14,14,Fernando,Alonso,aston_martin,Aston Martin,6,9,9,2,Finished,57,5579629,3,48,1:34.199
9,2024,1,Bahrain Grand Prix,2024-03-02,bahrain,stroll,STR,18,18,Lance,Stroll,aston_martin,Aston Martin,12,10,10,1,Finished,57,5597958,16,30,1:35.632


In [9]:
print("Rows (drivers):", len(df_r1))
print("Exactly one winner:", (df_r1["position"] == 1).sum() == 1)
print("Positions are 1..N with no gaps:", sorted(df_r1["position"]) == list(range(1, len(df_r1) + 1)))
print("Unique driver_id:", df_r1["driver_id"].is_unique)
print("Missing values per column (only columns with any):")
print(df_r1.isna().sum()[lambda s: s > 0].to_string() or "  none")

Rows (drivers): 20
Exactly one winner: True
Positions are 1..N with no gaps: True
Unique driver_id: True
Missing values per column (only columns with any):
Series([], )


## 4. Qualifying: 2024 Round 1

In [10]:
rq = get("/2024/1/qualifying.json", params={"limit": 100}, save_as="2024_r01_qualifying.json")
quali_r1 = rq.json()
q_race = quali_r1["MRData"]["RaceTable"]["Races"][0]
print("Status:", rq.status_code, "| MRData:", mrdata_summary(quali_r1))
print("Race keys:", list(q_race.keys()))
print("Record list key: 'QualifyingResults' | first record keys:", list(q_race["QualifyingResults"][0].keys()))

quali_rows = []
for rec in q_race["QualifyingResults"]:
    quali_rows.append({
        "driver_id": rec["Driver"]["driverId"], "driver_code": rec["Driver"].get("code"),
        "car_number": rec.get("number"), "constructor_id": rec["Constructor"]["constructorId"],
        "quali_position": int(rec["position"]),
        "Q1": rec.get("Q1"), "Q2": rec.get("Q2"), "Q3": rec.get("Q3"),
    })
df_q1 = pd.DataFrame(quali_rows)
df_q1

Status: 200 | MRData: {'xmlns': '', 'series': 'f1', 'url': 'https://api.jolpi.ca/ergast/f1/2024/1/qualifying.json', 'limit': '100', 'offset': '0', 'total': '20'}
Race keys: ['season', 'round', 'url', 'raceName', 'Circuit', 'date', 'time', 'QualifyingResults']
Record list key: 'QualifyingResults' | first record keys: ['number', 'position', 'Driver', 'Constructor', 'Q1', 'Q2', 'Q3']


,driver_id,driver_code,car_number,constructor_id,quali_position,Q1,Q2,Q3
0,max_verstappen,VER,1,red_bull,1,1:30.031,1:29.374,1:29.179
1,leclerc,LEC,16,ferrari,2,1:30.243,1:29.165,1:29.407
2,russell,RUS,63,mercedes,3,1:30.350,1:29.922,1:29.485
3,sainz,SAI,55,ferrari,4,1:29.909,1:29.573,1:29.507
4,perez,PER,11,red_bull,5,1:30.221,1:29.932,1:29.537
5,alonso,ALO,14,aston_martin,6,1:30.179,1:29.801,1:29.542
6,norris,NOR,4,mclaren,7,1:30.143,1:29.941,1:29.614
7,piastri,PIA,81,mclaren,8,1:30.531,1:30.122,1:29.683
8,hamilton,HAM,44,mercedes,9,1:30.451,1:29.718,1:29.710
9,hulkenberg,HUL,27,haas,10,1:30.566,1:29.851,1:30.502


In [11]:
print("Drivers in qualifying:", len(df_q1))
print("Have Q1 / Q2 / Q3:", df_q1["Q1"].notna().sum(), "/", df_q1["Q2"].notna().sum(), "/", df_q1["Q3"].notna().sum())
print("Same driver_id set as race results:", set(df_q1["driver_id"]) == set(df_r1["driver_id"]))
# Qualifying position vs race grid (they can differ because of grid penalties)
cmp = df_q1.merge(df_r1[["driver_id", "grid"]], on="driver_id")
diff = cmp[cmp["quali_position"] != cmp["grid"]]
print("Drivers whose grid differs from qualifying position:", len(diff))
diff

Drivers in qualifying: 20
Have Q1 / Q2 / Q3: 20 / 15 / 10
Same driver_id set as race results: True
Drivers whose grid differs from qualifying position: 0


,driver_id,driver_code,car_number,constructor_id,quali_position,Q1,Q2,Q3,grid


## 5. Sprint

We do not guess which 2024 rounds had sprints. We read the 2024 schedule and pick the rounds whose entry contains a `Sprint` session.

In [12]:
rs = get("/2024/races.json", params={"limit": 100}, save_as="2024_schedule.json")
schedule_2024 = rs.json()["MRData"]["RaceTable"]["Races"]
sprint_rounds = [int(rc["round"]) for rc in schedule_2024 if "Sprint" in rc]
print("2024 races in schedule:", len(schedule_2024))
print("Rounds with a Sprint session:", sprint_rounds)
print("Schedule keys of a sprint weekend:", list(schedule_2024[sprint_rounds[0] - 1].keys()))
print("Schedule keys of a normal weekend:", list(schedule_2024[0].keys()))

2024 races in schedule: 24
Rounds with a Sprint session: [5, 6, 11, 19, 21, 23]
Schedule keys of a sprint weekend: ['season', 'round', 'url', 'raceName', 'Circuit', 'date', 'time', 'FirstPractice', 'Qualifying', 'Sprint', 'SprintQualifying']
Schedule keys of a normal weekend: ['season', 'round', 'url', 'raceName', 'Circuit', 'date', 'time', 'FirstPractice', 'SecondPractice', 'ThirdPractice', 'Qualifying']


In [13]:
SPRINT_ROUND = sprint_rounds[0]
rsp = get(f"/2024/{SPRINT_ROUND}/sprint.json", params={"limit": 100},
          save_as=f"2024_r{SPRINT_ROUND:02d}_sprint.json")
sprint_js = rsp.json()
s_race = sprint_js["MRData"]["RaceTable"]["Races"][0]
print("Status:", rsp.status_code, "| MRData:", mrdata_summary(sprint_js))
print("Race:", s_race["raceName"], "| race keys:", list(s_race.keys()))
print("Record list key: 'SprintResults' | first record keys:", list(s_race["SprintResults"][0].keys()))

df_sprint = flatten_results(sprint_js, record_key="SprintResults")
df_sprint[["driver_id", "driver_code", "constructor_id", "grid", "position", "position_text",
           "points", "status", "laps"]]

Status: 200 | MRData: {'xmlns': '', 'series': 'f1', 'url': 'https://api.jolpi.ca/ergast/f1/2024/5/sprint.json', 'limit': '100', 'offset': '0', 'total': '20'}
Race: Chinese Grand Prix | race keys: ['season', 'round', 'url', 'raceName', 'Circuit', 'date', 'time', 'SprintResults']
Record list key: 'SprintResults' | first record keys: ['number', 'position', 'positionText', 'points', 'Driver', 'Constructor', 'grid', 'laps', 'status', 'Time', 'FastestLap']


,driver_id,driver_code,constructor_id,grid,position,position_text,points,status,laps
0,max_verstappen,VER,red_bull,4,1,1,8,Finished,19
1,hamilton,HAM,mercedes,2,2,2,7,Finished,19
2,perez,PER,red_bull,6,3,3,6,Finished,19
3,leclerc,LEC,ferrari,7,4,4,5,Finished,19
4,sainz,SAI,ferrari,5,5,5,4,Finished,19
5,norris,NOR,mclaren,1,6,6,3,Finished,19
6,piastri,PIA,mclaren,8,7,7,2,Finished,19
7,russell,RUS,mercedes,11,8,8,1,Finished,19
8,zhou,ZHO,sauber,10,9,9,0,Finished,19
9,kevin_magnussen,MAG,haas,12,10,10,0,Finished,19


## 6. API behaviour and edge cases

### 6.1–6.3 Requests that should fail or return nothing

In [14]:
def describe(label, path, params=None):
    resp = get(path, params=params)
    row = {"case": label, "request": resp.url.replace(BASE_URL, ""), "status": resp.status_code,
           "content_type": resp.headers.get("Content-Type", "")[:30]}
    try:
        js = resp.json()
        info = mrdata_summary(js) if "MRData" in js else {}
        races = js.get("MRData", {}).get("RaceTable", {}).get("Races")
        row.update({"limit": info.get("limit"), "offset": info.get("offset"), "total": info.get("total"),
                    "races_returned": None if races is None else len(races),
                    "records": None if races is None else sum(len(x.get("Results", [])) for x in races),
                    "body_start": "" if "MRData" in js else str(js)[:60]})
    except ValueError:
        row["body_start"] = resp.text[:60].replace("\n", " ")
    return row

edge_rows = [
    describe("1. round that does not exist", "/2024/99/results.json"),
    describe("1b. season with no F1 data", "/1900/results.json"),
    describe("2. non-numeric round", "/2024/abc/results.json"),
    describe("2b. misspelled endpoint", "/2024/1/resultz.json"),
    describe("3. limit=0", "/2024/1/results.json", {"limit": 0}),
    describe("3. limit=-5", "/2024/1/results.json", {"limit": -5}),
    describe("3. limit=abc", "/2024/1/results.json", {"limit": "abc"}),
    describe("3. offset beyond total", "/2024/results.json", {"limit": 100, "offset": 100000}),
    describe("3. offset=-1", "/2024/1/results.json", {"offset": -1}),
]
df_edge = pd.DataFrame(edge_rows)
df_edge

,case,request,status,content_type,limit,offset,total,races_returned,records,body_start
0,1. round that does not exist,/2024/99/results.json,200,application/json,30,0,0,0.0,0.0,
1,1b. season with no F1 data,/1900/results.json,200,application/json,30,0,0,0.0,0.0,
2,2. non-numeric round,/2024/abc/results.json,404,text/html; charset=utf-8,NaN,NaN,NaN,NaN,NaN,"<!doctype html> <html lang=""en""> <head> <ti..."
3,2b. misspelled endpoint,/2024/1/resultz.json,400,application/json,None,None,None,NaN,NaN,{'detail': 'Bad Request: Endpoint does not sup...
4,3. limit=0,/2024/1/results.json?limit=0,200,application/json,30,0,20,1.0,20.0,
5,3. limit=-5,/2024/1/results.json?limit=-5,200,application/json,30,0,20,1.0,20.0,
6,3. limit=abc,/2024/1/results.json?limit=abc,200,application/json,30,0,20,1.0,20.0,
7,3. offset beyond total,/2024/results.json?limit=100&offset=100000,200,application/json,100,100000,479,0.0,0.0,
8,3. offset=-1,/2024/1/results.json?offset=-1,200,application/json,30,0,20,1.0,20.0,


### 6.4 Do fields differ between results, qualifying and sprint records?

In [15]:
def all_record_keys(js, record_key):
    keys = set()
    for rc in js["MRData"]["RaceTable"]["Races"]:
        for rec in rc.get(record_key, []):
            keys |= set(rec.keys())
    return keys

k_results = all_record_keys(results_r1, "Results")
k_quali = all_record_keys(quali_r1, "QualifyingResults")
k_sprint = all_record_keys(sprint_js, "SprintResults")
print("Common to all three      :", sorted(k_results & k_quali & k_sprint))
print("Results only (vs sprint) :", sorted(k_results - k_sprint))
print("Sprint only (vs results) :", sorted(k_sprint - k_results))
print("Qualifying only          :", sorted(k_quali - k_results))
print("Results but not quali    :", sorted(k_results - k_quali))

Common to all three      : ['Constructor', 'Driver', 'number', 'position']
Results only (vs sprint) : []
Sprint only (vs results) : []
Qualifying only          : ['Q1', 'Q2', 'Q3']
Results but not quali    : ['FastestLap', 'Time', 'grid', 'laps', 'points', 'positionText', 'status']


### 6.5 Do all driver records contain the same fields? (whole 2024 season)

In [16]:
from collections import Counter

season_records = [rec for js in season_pages for rc in js["MRData"]["RaceTable"]["Races"] for rec in rc["Results"]]
n = len(season_records)
result_field_counts = Counter(k for rec in season_records for k in rec.keys())
driver_field_counts = Counter(k for rec in season_records for k in rec["Driver"].keys())

presence = pd.DataFrame(
    [("result", k, c) for k, c in result_field_counts.items()] +
    [("Driver", k, c) for k, c in driver_field_counts.items()],
    columns=["level", "field", "records_with_field"])
presence["share"] = (presence["records_with_field"] / n).round(3)
print("Total 2024 result records:", n)
presence.sort_values(["level", "share", "field"], ascending=[True, True, True]).reset_index(drop=True)

Total 2024 result records: 479


,level,field,records_with_field,share
0,Driver,code,479,1.000
1,Driver,dateOfBirth,479,1.000
2,Driver,driverId,479,1.000
3,Driver,familyName,479,1.000
4,Driver,givenName,479,1.000
5,Driver,nationality,479,1.000
6,Driver,permanentNumber,479,1.000
7,Driver,url,479,1.000
8,result,Time,432,0.902
9,result,FastestLap,462,0.965


In [17]:
df_season = pd.concat([flatten_results(js) for js in season_pages], ignore_index=True)
no_time = df_season[df_season["race_time_millis"].isna()]
print("Records without 'Time':", len(no_time), "→ their statuses:", no_time["status"].value_counts().to_dict())
no_fl = df_season[df_season["fastest_lap_rank"].isna()]
print("Records without 'FastestLap':", len(no_fl), "→ their statuses:", no_fl["status"].value_counts().to_dict())

Records without 'Time': 47 → their statuses: {'Retired': 43, 'Did not start': 3, 'Disqualified': 1}
Records without 'FastestLap': 17 → their statuses: {'Retired': 13, 'Did not start': 3, 'Disqualified': 1}


### 6.6–6.7 How are retirements (DNF) represented, and what is `position` for non-finishers?

In [18]:
print("positionText values that are not numbers:",
      sorted(df_season.loc[~df_season["position_text"].str.isdigit(), "position_text"].unique()))
print("Rows with a missing numeric position:", df_season["position"].isna().sum())
print()
print("status vs positionText (2024 season):")
status_table = pd.crosstab(df_season["status"],
                           df_season["position_text"].where(~df_season["position_text"].str.isdigit(), "number"))
status_table

positionText values that are not numbers: ['D', 'R', 'W']
Rows with a missing numeric position: 0

status vs positionText (2024 season):


position_text,D,R,W,number
status,,,,
Did not start,0,0,3,0
Disqualified,1,0,0,1
Finished,0,0,0,287
Lapped,0,0,0,138
Retired,0,43,0,6


In [19]:
# Look inside a few races: are non-finishers placed after all finishers, ordered by laps completed?
checks = []
for rnd, g in df_season.groupby("round"):
    g = g.sort_values("position")
    finished = g["position_text"].str.isdigit()
    dnf_after_finishers = (not finished.any()) or (not (~finished).any()) or \
        g.loc[~finished, "position"].min() > g.loc[finished, "position"].max()
    laps_non_increasing = g.loc[~finished, "laps"].is_monotonic_decreasing
    checks.append({"round": rnd, "non_finishers": int((~finished).sum()),
                   "non_finishers_after_finishers": dnf_after_finishers,
                   "non_finishers_ordered_by_laps_desc": laps_non_increasing})
df_order = pd.DataFrame(checks)
print("Races where non-finishers are all placed after finishers:",
      df_order["non_finishers_after_finishers"].sum(), "of", len(df_order))
print("Races where non-finishers are ordered by laps (descending):",
      df_order["non_finishers_ordered_by_laps_desc"].sum(), "of", len(df_order))

example_round = df_order.loc[df_order["non_finishers"].idxmax(), "round"]
print(f"\nExample: round {example_round}, the bottom of the classification")
df_season[df_season["round"] == example_round].sort_values("position")[
    ["round", "driver_code", "grid", "position", "position_text", "laps", "status"]].tail(6)

Races where non-finishers are all placed after finishers: 23 of 24
Races where non-finishers are ordered by laps (descending): 24 of 24

Example: round 9, the bottom of the classification


,round,driver_code,grid,position,position_text,laps,status
173,9,ZHO,20,15,15,69,Lapped
174,9,SAI,12,16,R,52,Retired
175,9,ALB,10,17,R,52,Retired
176,9,PER,16,18,R,51,Retired
177,9,LEC,11,19,R,40,Retired
178,9,SAR,13,20,R,23,Retired


**Special cases in detail.** Which race broke the "non-finishers after finishers" rule? And how do disqualified, non-starting and *classified* retired drivers look?

In [20]:
exception_rounds = df_order.loc[~df_order["non_finishers_after_finishers"], "round"].tolist()
print("Round(s) where a numbered driver is placed below a non-finisher:", exception_rounds)
for rnd in exception_rounds:
    print(df_season[df_season["round"] == rnd].sort_values("position")[
        ["round", "race_name", "driver_code", "grid", "position", "position_text", "laps", "status"]].tail(3).to_string())

special = df_season[df_season["status"].isin(["Disqualified", "Did not start"]) |
                    ((df_season["status"] == "Retired") & df_season["position_text"].str.isdigit())]
print("\nDisqualified, did-not-start, and retired-but-classified records (2024):")
special[["round", "race_name", "driver_code", "grid", "position", "position_text", "laps", "status"]]

Round(s) where a numbered driver is placed below a non-finisher: [14]
     round           race_name driver_code  grid  position position_text  laps        status
276     14  Belgian Grand Prix         HUL    16        18            18    44      Finished
277     14  Belgian Grand Prix         ZHO    19        19             R     5       Retired
278     14  Belgian Grand Prix         RUS     6        20            20    44  Disqualified

Disqualified, did-not-start, and retired-but-classified records (2024):


,round,race_name,driver_code,grid,position,position_text,laps,status
56,3,Australian Grand Prix,RUS,7,17,17,56,Retired
238,12,British Grand Prix,GAS,19,20,W,0,Did not start
278,14,Belgian Grand Prix,RUS,6,20,20,44,Disqualified
335,17,Azerbaijan Grand Prix,PER,4,17,17,49,Retired
336,17,Azerbaijan Grand Prix,SAI,3,18,18,49,Retired
337,17,Azerbaijan Grand Prix,STR,13,19,19,45,Retired
357,18,Singapore Grand Prix,MAG,14,19,19,57,Retired
416,21,São Paulo Grand Prix,ALB,7,18,W,0,Did not start
417,21,São Paulo Grand Prix,STR,10,19,W,0,Did not start
418,21,São Paulo Grand Prix,HUL,18,20,D,0,Disqualified


### 6.8 Does `grid = 0` occur, and what does it appear to mean?

In [21]:
grid0 = df_season[df_season["grid"] == 0]
print("2024 records with grid = 0:", len(grid0))
grid0[["round", "race_name", "driver_id", "constructor_id", "grid", "position", "position_text", "laps", "status"]]

2024 records with grid = 0: 1


,round,race_name,driver_id,constructor_id,grid,position,position_text,laps,status
391,20,Mexico City Grand Prix,ocon,alpine,0,13,13,70,Lapped


If `grid = 0` rows exist, we check them against **qualifying for the same race** (at most 3 races, to limit requests). If the driver has a qualifying position but `grid = 0`, then 0 does *not* mean "did not qualify". It is consistent with a **pit-lane start**, but the API itself never states the reason.

In [22]:
grid0_check = []
for rnd in sorted(grid0["round"].unique())[:3]:
    qjs = get(f"/2024/{int(rnd)}/qualifying.json", params={"limit": 100},
              save_as=f"2024_r{int(rnd):02d}_qualifying.json").json()
    qpos = {rec["Driver"]["driverId"]: int(rec["position"])
            for rc in qjs["MRData"]["RaceTable"]["Races"] for rec in rc["QualifyingResults"]}
    for _, row in grid0[grid0["round"] == rnd].iterrows():
        grid0_check.append({"round": int(rnd), "driver_id": row["driver_id"],
                            "qualifying_position": qpos.get(row["driver_id"]),
                            "race_status": row["status"], "laps": row["laps"]})
pd.DataFrame(grid0_check) if grid0_check else "No grid = 0 rows in 2024"

,round,driver_id,qualifying_position,race_status,laps
0,20,ocon,19,Lapped,70


### 6.9 Are driver IDs stable?

In [23]:
# Within 2024: does each driver_id always map to one code / permanent number / name?
per_driver = df_season.groupby("driver_id").agg(
    codes=("driver_code", "nunique"), numbers=("driver_number", "nunique"),
    names=("driver_last_name", "nunique"), constructors=("constructor_id", "nunique"),
    races=("round", "count"))
print("2024 drivers:", len(per_driver))
print("Any driver_id with more than one code/number/name:",
      bool((per_driver[["codes", "numbers", "names"]] > 1).any().any()))
print("Drivers who raced for more than one constructor in 2024:")
print(per_driver[per_driver["constructors"] > 1][["constructors", "races"]].to_string() or "  none")

2024 drivers: 24
Any driver_id with more than one code/number/name: False
Drivers who raced for more than one constructor in 2024:
           constructors  races
driver_id                     
bearman               2      3


**Is the `Driver` object historical or current?** The result's `number` is the car number used in *that* race. The driver's `permanentNumber` comes from the driver record. If they differ, the driver record reflects the driver's **current** details, not the details at the time of the race.

In [24]:
number_mismatch = df_r1[df_r1["driver_number"] != df_r1["car_number"]]
print("2024 R1 drivers whose permanentNumber differs from the car number they raced with:")
number_mismatch[["driver_id", "driver_code", "driver_number", "car_number"]]

2024 R1 drivers whose permanentNumber differs from the car number they raced with:


,driver_id,driver_code,driver_number,car_number
0,max_verstappen,VER,3,1
5,norris,NOR,1,4


In [25]:
# Across seasons: the same person should keep the same driverId in 2023 and 2024.
def season_drivers(season):
    js = get(f"/{season}/drivers.json", params={"limit": 100},
             save_as=f"{season}_drivers.json").json()
    return pd.DataFrame(js["MRData"]["DriverTable"]["Drivers"])

d23, d24 = season_drivers(2023), season_drivers(2024)
person = ["givenName", "familyName", "dateOfBirth"]
both = d23.merge(d24, on=person, suffixes=("_2023", "_2024"))
print("Drivers present in both seasons:", len(both))
print("Same driverId in both seasons:", (both["driverId_2023"] == both["driverId_2024"]).all())

# One long career: Lewis Hamilton's results are all under a single ID
ham = get("/drivers/hamilton/results.json", params={"limit": 1}).json()
first_ham = ham["MRData"]["RaceTable"]["Races"][0]
print("driverId 'hamilton' → total race results:", ham["MRData"]["total"],
      "| first:", first_ham["season"], first_ham["raceName"])

Drivers present in both seasons: 21
Same driverId in both seasons: True


driverId 'hamilton' → total race results: 394 | first: 2007 Australian Grand Prix


### 6.10 Are constructor IDs and names stable?

In [26]:
# Within 2024: does each constructor_id always have the same name?
names_per_id = df_season.groupby("constructor_id")["constructor_name"].unique()
print("2024 constructor_id → name(s):")
print(names_per_id.to_string())

def season_constructors(season):
    js = get(f"/{season}/constructors.json", params={"limit": 100},
             save_as=f"{season}_constructors.json").json()
    return pd.DataFrame(js["MRData"]["ConstructorTable"]["Constructors"])[["constructorId", "name"]]

c23, c24 = season_constructors(2023), season_constructors(2024)
print("\nIDs only in 2023:", sorted(set(c23["constructorId"]) - set(c24["constructorId"])))
print("IDs only in 2024:", sorted(set(c24["constructorId"]) - set(c23["constructorId"])))
print("IDs in both      :", sorted(set(c23["constructorId"]) & set(c24["constructorId"])))

2024 constructor_id → name(s):
constructor_id
alpine          [Alpine F1 Team]
aston_martin      [Aston Martin]
ferrari                [Ferrari]
haas              [Haas F1 Team]
mclaren                [McLaren]
mercedes              [Mercedes]
rb                  [RB F1 Team]
red_bull              [Red Bull]
sauber                  [Sauber]
williams              [Williams]



IDs only in 2023: ['alfa', 'alphatauri']
IDs only in 2024: ['rb', 'sauber']
IDs in both      : ['alpine', 'aston_martin', 'ferrari', 'haas', 'mclaren', 'mercedes', 'red_bull', 'williams']


## 7. Response headers and rate limits

In [27]:
r_headers = get("/2024/1/results.json").headers
print("All response headers (2024/1/results):")
for name, value in r_headers.items():
    print(f"  {name}: {value[:90]}")

rate_related = {k: v for k, v in r_headers.items()
                if any(word in k.lower() for word in ["rate", "retry", "limit", "remaining"])}
print("\nHeaders mentioning rate/retry/limit/remaining:", rate_related or "NONE")

All response headers (2024/1/results):
  Date: Fri, 25 Sep 2026 09:29:17 GMT
  Content-Type: application/json
  Transfer-Encoding: chunked
  Connection: keep-alive
  Age: 371
  allow: GET, HEAD, OPTIONS
  Cache-Control: max-age=600
  Content-Encoding: gzip
  cross-origin-opener-policy: same-origin
  expires: Fri, 25 Sep 2026 09:23:21 GMT
  referrer-policy: same-origin
  Server: cloudflare
  vary: Accept, origin, accept-encoding
  x-content-type-options: nosniff
  x-frame-options: DENY
  last-modified: Fri, 25 Sep 2026 09:23:05 GMT
  cf-cache-status: HIT
  Nel: {"report_to":"cf-nel","success_fraction":0.0,"max_age":604800}
  Report-To: {"group":"cf-nel","max_age":604800,"endpoints":[{"url":"https://a.nel.cloudflare.com/repor
  CF-RAY: a40900eda9cc0769-DEL
  alt-svc: h3=":443"; ma=86400

Headers mentioning rate/retry/limit/remaining: NONE


**Official documentation** (checked 2026-09-25, [jolpica-f1 docs/rate_limits.md](https://github.com/jolpica/jolpica-f1/blob/main/docs/rate_limits.md) and [docs/README.md](https://github.com/jolpica/jolpica-f1/blob/main/docs/README.md)):

| Item | Documented value |
|---|---|
| Burst limit (unauthenticated) | 4 requests per second |
| Sustained limit (unauthenticated) | 500 requests per hour |
| When exceeded | HTTP **429** Too Many Requests ("Request was throttled") |
| Future | Limits "are subject to change, and **will decrease**" |
| `limit` parameter | Default 30, maximum 100 |
| `offset` parameter | Default 0 |

We did **not** deliberately trigger a 429 (that would be impolite). The Step 3 client must handle 429 anyway.

In [28]:
log = pd.DataFrame(REQUEST_LOG)
print("Requests made in this run:", len(log))
print("Status codes:", log["status"].value_counts().to_dict())
print("Server response time (s): median", log["seconds"].median(), "| max", log["seconds"].max())

Requests made in this run: 31
Status codes: {200: 29, 404: 1, 400: 1}
Server response time (s): median 0.11 | max 0.67


## 8. Raw responses saved

**Naming convention** for this exploration (folder `data/raw/api_exploration/`):
`{season}_r{round:02d}_{endpoint}.json` for one race, `{season}_season_results_offset_{offset:04d}.json` for season pages, `{season}_{endpoint}.json` for season lists.

Files contain the **exact bytes** the API returned. They are small, and `data/raw/` is git-ignored (re-creatable by running this notebook). The Step 4 collection will use its own folder `data/raw/jolpica/`.

In [29]:
saved = sorted(EXPLORATION_DIR.glob("*.json"))
pd.DataFrame({"file": [p.relative_to(PROJECT_ROOT).as_posix() for p in saved],
              "size_kb": [round(p.stat().st_size / 1024, 1) for p in saved]})

,file,size_kb
0,data/raw/api_exploration/2023_constructors.json,1.4
1,data/raw/api_exploration/2023_drivers.json,4.6
2,data/raw/api_exploration/2024_constructors.json,1.4
3,data/raw/api_exploration/2024_drivers.json,5.2
4,data/raw/api_exploration/2024_r01_qualifying.json,8.9
5,data/raw/api_exploration/2024_r01_results.json,12.6
6,data/raw/api_exploration/2024_r05_sprint.json,11.8
7,data/raw/api_exploration/2024_r20_qualifying.json,8.9
8,data/raw/api_exploration/2024_schedule.json,14.6
9,data/raw/api_exploration/2024_season_results_o...,61.9


## 9. Findings (verified on 2026-09-25)

Every statement below was observed in this notebook's outputs, unless it is labelled **Interpretation** or **Open**.

### 9.1 Base URL and endpoints

| Item | Verified |
|---|---|
| Base URL | `https://api.jolpi.ca/ergast/f1` (from `config.yaml`), JSON, no API key |
| Race results | `/{season}/{round}/results.json` and `/{season}/results.json` → `RaceTable.Races[].Results[]` |
| Qualifying | `/{season}/{round}/qualifying.json` → `Races[].QualifyingResults[]` |
| Sprint | `/{season}/{round}/sprint.json` → `Races[].SprintResults[]` |
| Schedule | `/{season}/races.json` → `Races[]`. Sprint weekends have `Sprint` and `SprintQualifying` keys. 2024 sprint rounds: 5, 6, 11, 19, 21, 23 |
| Season lists | `/{season}/drivers.json` → `DriverTable.Drivers[]`; `/{season}/constructors.json` → `ConstructorTable.Constructors[]` |
| Driver filter | `/drivers/{driverId}/results.json` works |

### 9.2 Pagination and `MRData.total`

- `limit` is respected (10 → 10, 20 → 20, 100 → 100 records). The **default is 30**. `limit=1000` is **silently capped at 100**.
- `offset` moves the window exactly: two pages of 10 equal one page of 20, in the same order, with no overlap.
- `MRData.total` is the number of **result records** (not races) matching the query. It stays the same on every page (479 for 2024).
- Paging 2024 with `limit=100` took 5 pages and collected 479 records = `total`, with **0 duplicates**. The paging is safe.
- **Races are split across pages** (2024 rounds 6, 11, 16 and 21), so records must be grouped by (season, round) after collection.
- **All values are strings**, including `limit`, `offset`, `total`, `position`, `grid`, `points` and `laps`.

### 9.3 Fields per endpoint

| Endpoint | Record fields |
|---|---|
| Results | number, position, positionText, points, Driver, Constructor, grid, laps, status, Time*, FastestLap* |
| Sprint | **identical field set to results** |
| Qualifying | number, position, Driver, Constructor, Q1, Q2*, Q3* (no grid, points, status, laps) |

\* not always present. In 2024 results, `Time` is missing in 47 of 479 records and `FastestLap` in 17 (retired, did-not-start and disqualified drivers). In 2024 R1 qualifying, Q1/Q2/Q3 were present for 20/15/10 drivers (knock-out format). Driver fields (driverId, permanentNumber, code, givenName, familyName, dateOfBirth, nationality, url) were present in **100%** of 2024 records. Constructor fields: constructorId, url, name, nationality.

### 9.4 DNF / status representation (2024 season)

- `position` is **always a number**, for every driver, including retired, non-starting and disqualified drivers. Non-finishers are ordered by **laps completed, descending**, in 24 of 24 races.
- Non-numeric `positionText` values seen in 2024: **R** (Retired, 43), **W** (status "Did not start", 3), **D** (Disqualified, 1). E, F and N were not seen in 2024.
- Status values in 2024: Finished, Lapped, Retired, Did not start, Disqualified.
- **Retired but classified:** 6 records have status "Retired" but a **numeric** positionText (e.g. RUS in Australia, 56 laps).
- **Disqualification is not always "D":** at the 2024 Belgian GP, Russell (disqualified) has positionText **"20"**, position 20, placed below a retirement. That is the single exception to "non-finishers after finishers".
- **Did not start:** positionText "W", laps 0, but a **non-zero grid** (e.g. GAS British GP, grid 19).

### 9.5 `grid = 0`

- 1 record in 2024: Ocon, 2024 Mexico City GP, grid 0, qualified 19th, finished 13th (70 laps).
- **Interpretation:** consistent with a **pit-lane start** (he qualified, so 0 does not mean "did not qualify"). The API does not state the reason. **Open:** meaning in older seasons, to be checked on full history at Step 7.

### 9.6 Driver IDs

- In 2024 each driverId maps to exactly one code, number and name. 21 drivers raced in both 2023 and 2024 with the **same driverId**. `hamilton` covers 394 results from the 2007 Australian GP onward under one ID.
- One driver raced for two constructors in 2024 (bearman, 3 races).
- The `Driver` object seems to show **current** details, not race-time details. Norris shows permanentNumber 1 but raced car 4 in 2024 R1; Verstappen shows 3 but raced car 1. **Interpretation:** use the result's `number` for the car number at race time. `permanentNumber` is not historical.

### 9.7 Constructor IDs

- Within 2024, each constructorId has exactly one name.
- **Renamed teams get new IDs:** `alfa` (2023) → `sauber` (2024), `alphatauri` (2023) → `rb` (2024). The other 8 IDs are unchanged. This confirms the blueprint's lineage concern (§17) for 2023→2024.

### 9.8 Edge cases

| Request | Result |
|---|---|
| Round that does not exist (`/2024/99/`) | **HTTP 200**, `total = 0`, empty `Races`. No error! |
| Season with no data (`/1900/`) | **HTTP 200**, `total = 0` |
| Non-numeric round (`/2024/abc/`) | HTTP **404**, HTML page (not JSON) |
| Misspelled endpoint (`/resultz`) | HTTP **400**, JSON `{"detail": "Bad Request: ..."}` |
| `limit=0`, `-5` or `abc` | HTTP 200, silently falls back to **30** |
| `offset=-1` | HTTP 200, treated as **0** |
| `offset` beyond total | HTTP 200, 0 records, `total` still 479 |

**Consequence for Step 3:** a 200 status does not mean data was found. The client must check `total` / record counts, and it cannot assume every error body is JSON.

### 9.9 Headers and rate limits

- Headers: `Content-Type: application/json`, `Cache-Control: max-age=600`, `Server: cloudflare`, `cf-cache-status` (HIT/MISS), `Age`, gzip encoding.
- **No rate-limit or retry headers** (nothing containing rate, retry, limit or remaining).
- Documented limits (official docs, see Part 7): **4 req/s burst, 500 req/h sustained**, HTTP 429 when exceeded, "will decrease" in future. This run made 31 requests (29 × 200, 1 × 404, 1 × 400), median response 0.11 s.

### 9.10 Raw files

14 small JSON files in `data/raw/api_exploration/` (about 370 KB), naming convention in Part 8. The folder is git-ignored and re-created by running this notebook.

### 9.11 Items for later steps (not changed here)

| Item | Why it matters | Step |
|---|---|---|
| DNF rule: blueprint §14 uses positionText R/N, but 6 "Retired" drivers are **classified** with numeric positionText | Decide whether classified retirements count as DNF | 7 (decision needed) |
| DSQ detection: a disqualification can have a numeric positionText | Need `status` as well as positionText | 7 |
| "Did not start" = positionText W **with a grid value** | The starters filter must use status/positionText, not grid | 7–8 |
| positionText codes E, F, N not observed in 2024 | Still assumptions | 7 |
| grid = 0 in older seasons | Only one 2024 case seen | 7 |
| Driver object reflects current details | Don't use permanentNumber as a historical feature | 8 |
| Constructor renames before 2023 | Only 2023→2024 checked | 11 |
| Actual throttling behaviour (429) | Not triggered on purpose | 3 |